## Structured Output

Models can be requested to provide their reponse in a format matching a given schema. This is useful for ensuring that the output can be easily parsed and used in a subsequent processing. LangChain supports multiple schema types and methods for enforcing strctured output.

### Pydantic

Pydantic models provide richest feature set with field validation, descriptions and nested strcutures.

In [1]:
import os
from dotenv import load_dotenv
import anthropic
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = os.getenv("ANTHROPIC_API_KEY")

model = init_chat_model("claude-sonnet-5")

In [2]:
from pydantic import BaseModel, Field

# The model class for the response structure that we want. The class inherits from the BaseModel and the Field has numerous properties that can be used with the fields of the model
class Movie(BaseModel):
  title:str = Field(description="This is the title of the movie")
  year:str = Field(description="This is the year the movie was released")
  director:str = Field(description="The director ot the movie")
  rating:str = Field(description="The movie rating out of 10")

In [3]:
model_with_structuredOutput = model.with_structured_output(Movie)

In [4]:
# unstructured output
response = model.invoke("Provide details of the movie inception")


# strcutured output
response01 = model_with_structuredOutput.invoke("Provide details of the movie inception")

In [5]:
response


AIMessage(content='# Inception (2010)\n\n## Overview\n**Inception** is a science fiction action thriller written and directed by Christopher Nolan. The film explores dream-sharing technology and is known for its complex, layered narrative structure.\n\n## Key Details\n- **Release Date:** July 16, 2010\n- **Director:** Christopher Nolan\n- **Writer:** Christopher Nolan\n- **Runtime:** 148 minutes\n- **Budget:** ~$160 million\n- **Box Office:** Over $836 million worldwide\n\n## Cast\n- **Leonardo DiCaprio** as Dom Cobb\n- **Joseph Gordon-Levitt** as Arthur\n- **Elliot Page (Ellen Page)** as Ariadne\n- **Tom Hardy** as Eames\n- **Ken Watanabe** as Saito\n- **Marion Cotillard** as Mal\n- **Cillian Murphy** as Robert Fischer\n- **Michael Caine** as Professor Stephen Miles\n\n## Plot Summary\nDom Cobb is a skilled thief who specializes in "extraction" — stealing secrets from people\'s subconscious during dream states. Offered a chance to have his criminal history erased, Cobb must perform "i

In [6]:
response01

Movie(title='Inception', year='2010', director='Christopher Nolan', rating='8.8')

### Message Output alongside parsed strcuture

In [ ]:
# include raw data along with the parsed structure. The 'include_raw=True', gives us the raw data send my the model an also strcutures it accordingly

model_with_structuredOutput02 = model.with_structured_output(Movie, include_raw=True)

response02 = model_with_structuredOutput02.invoke("Give me some information about the movie Inception")

response02

{'raw': AIMessage(content=[{'id': 'toolu_01TcCNUYNbtbJSeSQk5HDnVL', 'caller': {'type': 'direct'}, 'input': {'title': 'Inception', 'year': '2010', 'director': 'Christopher Nolan', 'rating': '8.8'}, 'name': 'Movie', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CfgkhcFSb2BFYZZvpr5oW', 'container': None, 'model': 'claude-sonnet-5', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 653, 'output_tokens': 108, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard'}, 'diagnostics': None, 'model_name': 'claude-sonnet-5', 'model_provider': 'anthropic'}, id='lc_run--01a105ae-b9c7-7482-bfb7-04555e73b71c-0', tool_calls=[{'name': 'Movie', 'args': {'title': 'Inception', 'year': '2010

### Nested Structure

In [9]:

# creating nested types for the strucutured outputs

class Actor(BaseModel):
  name:str = Field(description="The name of the actor")
  role:str = Field(description="The role of the actor in the movie")

class MovieDetails(BaseModel):
  title:str
  year:str = Field(description="The year of release of the movie")
  cast:list[Actor]
  genres:list[str]
  budget: float | None = Field(description="The budget of the movie in million USD")

model_with_structuredOutput03 = model.with_structured_output(MovieDetails, include_raw=True)

response04 = model_with_structuredOutput03.invoke("Give me some details about the movie Inception")
response04

{'raw': AIMessage(content=[{'id': 'toolu_01Pgp8npJPnwf3B8XY4TVrPe', 'caller': {'type': 'direct'}, 'input': {'title': 'Inception', 'year': '2010', 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'}, {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'}, {'name': 'Ellen Page', 'role': 'Ariadne'}, {'name': 'Tom Hardy', 'role': 'Eames'}, {'name': 'Ken Watanabe', 'role': 'Saito'}], 'genres': ['Science Fiction', 'Action', 'Thriller'], 'budget': 160}, 'name': 'MovieDetails', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CfgmdQqc77XMCx1LkvreY', 'container': None, 'model': 'claude-sonnet-5', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 764, 'output_tokens': 272, 'output_tokens_details': {'thinking_tokens': 0}, 'se

### TypeDict

TypeDict provides a simple alternative using python's Built-in typing, ideal for when you dont need runtime validation. Here the type validation will not happen like it happens using the pydantic

In [10]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
  title: Annotated[str,...,"The title of the movie"]
  year: Annotated[int,...,"The year of release of the movie"]
  director: Annotated[str,...,"The director of the movie"]
  rating: Annotated[float,...,"The rating of the movie"]

model_withTypeDict=model.with_structured_output(MovieDict)
response05 = model_withTypeDict.invoke("Give me the details about the movie Avengers")
response05

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [12]:
## Nested class example using TypeDict. 

class ActorDict(TypedDict):
  name:str
  role:str

class MovieDetailsTypeDict(TypedDict):
  title:str
  year:int
  cast:list[ActorDict]
  genres:list[str]
  budget:float | None = Field(description="The budget of the movie in million USD")

In [13]:
model_withNestedTypeDict = model.with_structured_output(MovieDetailsTypeDict)
response06=model_withNestedTypeDict.invoke("Give me some details about the movie Avengers")
response06

{'title': 'Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'budget': 220000000}

In [14]:
model.profile

{'name': 'Claude Sonnet 5',
 'release_date': '2026-06-29',
 'last_updated': '2026-06-30',
 'open_weights': False,
 'max_input_tokens': 1000000,
 'max_output_tokens': 128000,
 'text_inputs': True,
 'image_inputs': True,
 'audio_inputs': False,
 'pdf_inputs': True,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': True,
 'temperature': False,
 'image_url_inputs': True,
 'pdf_tool_message': True,
 'image_tool_message': True,
 'tool_call_streaming': True,
 'reasoning_effort_levels': ['low', 'medium', 'high', 'xhigh', 'max'],
 'reasoning_effort_default': 'high'}

### DataClasses

DataClass is a class, typically containing mainly data, although there arent any restrictions. You create it using @dataclass decorator

Lets now see and example where we will see an create_agent example using pydantic, typeDict and Dataclass to see the comparision and better understand the concept.

In [18]:
# Using Pydantic

from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfoPydantic(BaseModel):
  name: str = Field(description="This contains the name")
  email : str = Field(description="This contains the email")
  phone : str = Field(description="This contains the phone number")

agent_for_pydantic = create_agent(
  model="claude-sonnet-5",
  response_format=ContactInfoPydantic # selecting the response format here with the response_format property
)

result_contact_pydantic = agent_for_pydantic.invoke({
  "messages":[{
    "role" :"user",
    "content" : "Extract information from John Doe, John@example.com, (555) 124-6794"
  }]
})

print(result_contact_pydantic["structured_response"])


name='John Doe' email='John@example.com' phone='(555) 124-6794'


In [17]:
# Using TypeDict

from typing_extensions import TypedDict

class ContactInfoTypeDict(TypedDict):
  name : str
  email : str
  phone : str

agent_for_typedict = create_agent(
  model = "claude-sonnet-5",
  response_format = ContactInfoTypeDict
)

result_contact_typedict = agent_for_typedict.invoke({
  "messages":[{
    "role" : "user",
    "content" : "Extract the information from Pritam Dey, pritam@example.com, 80917324717"
  }]
})

print(result_contact_typedict["structured_response"])

{'name': 'Pritam Dey', 'email': 'pritam@example.com', 'phone': '80917324717'}


In [19]:
# Using DataClass

from dataclasses import dataclass

@dataclass
class ContactInfoDataClass:
  """Contact Information for a person"""
  name : str # the name of a person
  email : str # the email of a person
  phone : str # the phone number of a person

agent_for_dataclass = create_agent(
  model="claude-sonnet-5",
  response_format=ContactInfoDataClass
)

result_contact_dataclass = agent_for_dataclass.invoke({
  "messages": [{
    "role" : "user",
    "content" : "Extract the information from Payal, payal@example.com, 8917237916"
  }]
})

print(result_contact_dataclass["structured_response"])

ContactInfoDataClass(name='Payal', email='payal@example.com', phone='8917237916')


Here we have seen the different response format types and how we can format the data that we are getting back from the model in 3 different manners usinmg the Pydantic, typeDict and DataClass